# TP 2 : Auto-attention

Le bigramme ne regarde que le caractère courant. Ici on construit, sur des tenseurs au hasard, le calcul qui mélange les tokens déjà vus. On part d'une moyenne du passé, puis les poids deviennent dépendants des données.

À chaque version, une ligne de poids dit combien le token courant prend chez les tokens précédents, et cette ligne somme à 1. Rien ne va vers le futur.

B = nombre de séquences, T = positions, C = taille du vecteur, dh = taille d'une tête.


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))  # 1 sur et sous la diagonale, 0 dans le futur
a = a / a.sum(1, keepdim=True)  # chaque ligne devient une moyenne
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)
print("sommes", a.sum(1).tolist())
# 2e ligne de a = [1/2, 1/2, 0], donc c[1] = moyenne des deux premiers vecteurs de b
print("à la main", ((b[0] + b[1]) / 2).tolist(), "produit", c[1].tolist())


tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
sommes [1.0, 1.0, 1.0]
à la main [4.0, 5.5] produit [4.0, 5.5]


## Étape 1 : moyenne à la main

Sans lancer le code, la deuxième sortie c2 vaut (b0 + b1) / 2. La deuxième ligne de a est [1/2, 1/2, 0], donc le troisième vecteur est ignoré. En Python c'est c[1], et le print confirme le calcul à la main.

Chaque ligne somme à 1 pour que la sortie reste une moyenne. Sans la division, une position tardive serait une grosse somme, seulement parce qu'elle a plus de passé.


In [2]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b_idx in range(B):
    for t in range(T):
        xprev = x[b_idx, :t + 1]  # (t+1, C), le passé y compris t
        if b_idx == 0 and t in (0, 3, 7):
            print(f"t={t}  xprev {tuple(xprev.shape)}")
        xbow[b_idx, t] = xprev.mean(0)  # moyenne sur les tokens, pas sur les features

print("xbow", tuple(xbow.shape))


t=0  xprev (1, 2)
t=3  xprev (4, 2)
t=7  xprev (8, 2)
xbow (4, 8, 2)


## Étape 2 : Moyenner le passé

Une fois le préfixe sorti, xprev a la forme (t+1, C). À t = 0 c'est (1, 2), à t = 3 c'est (4, 2), à t = 7 c'est (8, 2). La moyenne est sur dim=0 parce que cet axe est le temps. On moyenne les tokens, pas les coordonnées entre elles. La sortie xbow garde la forme de x, un vecteur par position.


In [3]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print("produit == boucle", torch.allclose(xbow, xbow2, atol=1e-6))

# même moyenne, écrite comme le fera l'attention : 0 partout, -inf dans le futur, softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print("softmax == boucle", torch.allclose(xbow, xbow3, atol=1e-6))
print("ligne t=3", [round(v, 2) for v in wei[3].tolist()])


produit == boucle True
softmax == boucle True
ligne t=3 [0.25, 0.25, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0]


## Étapes 3 et 4 : vectoriser

wei @ x marche avec une matrice (T, T) et un tenseur (B, T, C) parce que PyTorch étend la matrice sur le batch. Elle est vue comme (1, T, T), et le produit se fait sur les deux derniers axes.

Aux cases qui valent -inf, exp(-inf) = 0. Le softmax leur donne un poids nul et renormalise le reste. Comme les cases autorisées valaient toutes 0, chacune des t+1 positions reçoit 1/(t+1). La ligne t = 3 est quatre fois 0,25 puis des zéros. Le masque est le seul endroit où on interdit le futur. Si on l'oublie à l'entraînement, le modèle voit la fin de la phrase, alors qu'à la génération cette fin n'existe pas encore.


In [4]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key = nn.Linear(C, head_size, bias=False)     # K : comment le token se présente
query = nn.Linear(C, head_size, bias=False)   # Q : ce que la position cherche
value = nn.Linear(C, head_size, bias=False)   # V : l'info qu'on copie si on est choisi

k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5  # / sqrt(dh), sinon le softmax s'écrase
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
out = wei @ value(x)

print("q,k", tuple(q.shape), "wei", tuple(wei.shape), "out", tuple(out.shape))
print(wei[0].detach().round(decimals=2))
print("sommes", wei[0].sum(-1).round(decimals=3).tolist())
print("masse vers le futur", wei[0].triu(1).abs().sum().item())


q,k (4, 8, 16) wei (4, 8, 8) out (4, 8, 16)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3100, 0.2900, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.2200, 0.2400, 0.2100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2000, 0.1700, 0.1500, 0.3400, 0.0000, 0.0000, 0.0000],
        [0.1300, 0.2500, 0.1300, 0.1100, 0.3100, 0.0700, 0.0000, 0.0000],
        [0.1600, 0.2000, 0.1100, 0.1100, 0.1400, 0.1700, 0.1100, 0.0000],
        [0.0800, 0.1200, 0.1100, 0.1500, 0.1100, 0.1100, 0.1600, 0.1600]])
sommes [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
masse vers le futur 0.0


## Étape 5 : calculer des poids d'attention

La dernière ligne de wei[0], donc wei[0, -1], correspond au dernier token. Il peut regarder tout le bloc, lui compris, et personne après. Les sommes de lignes valent 1, et la masse au-dessus de la diagonale vaut 0 : pas de poids vers le futur. Les formes sont q, k, v en (B, T, dh), wei en (B, T, T), out en (B, T, dh).

On mélange v et pas k. k sert à se faire trouver, v est l'information qu'on emporte si on est choisi. Les poids ne sont plus égaux à 1/(t+1), ils dépendent de x. Là les projections sont encore aléatoires, le motif ne veut rien dire. C'est l'entraînement du TP 4 qui apprendra les trois matrices.


In [5]:
q_ = torch.randn(B, T, head_size)
k_ = torch.randn(B, T, head_size)
brut = q_ @ k_.transpose(-2, -1)
scale = brut * head_size ** -0.5
print(f"variance brute {brut.var():.2f}   (proche de dh = {head_size})")
print(f"variance / sqrt(dh) {scale.var():.2f}")

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("softmax", torch.softmax(z, dim=-1).round(decimals=3).tolist())
print("softmax x8", torch.softmax(z * 8, dim=-1).round(decimals=3).tolist())


variance brute 17.47   (proche de dh = 16)
variance / sqrt(dh) 1.09
softmax [0.19200000166893005, 0.14300000667572021, 0.23499999940395355, 0.14300000667572021, 0.28700000047683716]
softmax x8 [0.032999999821186066, 0.003000000026077032, 0.16200000047683716, 0.003000000026077032, 0.800000011920929]


## Étape 6 : Pourquoi diviser par la racine de dh

q dit ce que la position cherche, k comment elle se laisse trouver, v ce qu'on copie. Le produit scalaire additionne dh termes à peu près indépendants, donc sa variance grandit avec dh (ici autour de 16, puis autour de 1 après la division). Un softmax sur des scores trop grands s'écrase sur le maximum, comme z * 8, et le gradient disparaît. Diviser par √dh garde plusieurs tokens avec un vrai poids.
